In [8]:
from brf_lightning.data import BearingDataModule
import yaml

# load yaml file
def load_yaml(file_path):
    with open(file_path, 'r') as file:
        data = yaml.safe_load(file)
    return data


# load the config
def load_config(config_path):
    config = load_yaml(config_path)
    if 'data' not in config:
        raise ValueError("Config file must contain a 'data' section.")
    
    data_config = config['data']
    
    return BearingDataModule(**data_config["init_args"])

# Example usage
config_path = 'configs/bearing/thl/simple_brf_order_v2.yaml'  # Replace with your actual config file path
try:
    data_module = load_config(config_path)
    print("Data module loaded successfully.")
except Exception as e:
    print(f"Error loading config: {e}")

Data module loaded successfully.


In [9]:
data_module.prepare_data()
data_module.setup()

In [10]:
import torch
from collections import Counter

# 1. Make sure your DataModule has instantiated its datasets
data_module.setup()  

# 2. Grab the raw Dataset objects from each DataLoader
train_ds = data_module.train_dataloader().dataset
val_ds   = data_module.val_dataloader().dataset
test_ds  = data_module.test_dataloader().dataset

# 3. Define a helper to count labels
def label_distribution(dataset):
    # assume __getitem__ returns (x, y)
    labels = [int(y) for _, y in dataset]
    return Counter(labels)

# 4. Compute
train_dist = label_distribution(train_ds)
val_dist   = label_distribution(val_ds)
test_dist  = label_distribution(test_ds)

print("Train label counts:", train_dist)
print(" Val label counts:",   val_dist)
print("Test label counts:",   test_dist)


global μ=0.000, σ=0.154
Data split summary (windows per split):
split                  test  train    val
file       label rpm                     
fAII10.csv 1     10     0.0  980.0    0.0
fAII20.csv 1     20   989.0    0.0  989.0
fAII30.csv 1     30     0.0  980.0    0.0
fIII10.csv 2     10     0.0  980.0    0.0
fIII20.csv 2     20   999.0    0.0  999.0
fIII30.csv 2     30     0.0  980.0    0.0
n10.csv    0     10     0.0  980.0    0.0
n20.csv    0     20   999.0    0.0  999.0
n30.csv    0     30     0.0  980.0    0.0


ValueError: too many values to unpack (expected 2)

In [3]:
x, y, speed = next(iter(data_module.train_dataloader()))
print("RMS of first window:", x[0].square().mean().sqrt().item())

RMS of first window: 0.24999991059303284


In [3]:
data_config = load_yaml(config_path)["data"]["init_args"]

# check if the data module is set up correctly
# compare to the config
if data_module.train_ds is not None:
    print("Train dataset is set up correctly.")
if data_module.val_ds is not None:
    print("Validation dataset is set up correctly.")

# Check the number of samples in each dataset
print(f"Number of training samples: {len(data_module.train_ds)}")
print(f"Number of validation samples: {len(data_module.val_ds)}")
# Check the first sample in the training dataset
if data_module.train_ds:
    first_sample = data_module.train_ds[0]
    print(f"First training sample: {first_sample}")
    # Check if the sample has correct shape
    print(f"X shape: {first_sample[0].shape} expected: [{int(data_config['window_size'] * 8192)}, 1]")
    # Check if y is a 0-dim or 1-dim tensor with a single value
    y = first_sample[1]
    if (y.dim() == 0) or (y.dim() == 1 and y.numel() == 1):
        print("y is a single value tensor (label).")
    else:
        print("y is not a single value tensor!")
    print(f"y shape: {y.shape} expected: torch.Size([]) or torch.Size([1])")


Train dataset is set up correctly.
Validation dataset is set up correctly.
Number of training samples: 5880
Number of validation samples: 5378
First training sample: (tensor([[ 0.0911],
        [-0.0862],
        [-0.0669],
        [-0.0803],
        [ 0.1422],
        [-0.0413],
        [-0.1187],
        [-0.0237],
        [ 0.0508],
        [ 0.0120],
        [-0.1433],
        [-0.1227],
        [ 0.0392],
        [ 0.0179],
        [-0.0713],
        [-0.0972],
        [-0.0331],
        [-0.0095],
        [ 0.0129],
        [ 0.0082],
        [-0.1037],
        [-0.0285],
        [-0.0138],
        [ 0.0053],
        [-0.0388],
        [-0.1032],
        [-0.0088],
        [-0.0023],
        [-0.1090],
        [-0.0286],
        [-0.0898],
        [ 0.1181],
        [ 0.0760],
        [-0.0387],
        [-0.1421],
        [ 0.0153],
        [ 0.1008],
        [-0.0122],
        [-0.0579],
        [-0.0432],
        [-0.0457],
        [ 0.0224],
        [-0.0361],
        [-0.0407

In [11]:
import torch

splits = {
    "train": data_module.train_ds,
    "val":   data_module.val_ds,
    "test":  data_module.test_ds
}

for split_name, ds in splits.items():
    if ds is None:
        print(f"{split_name}: None")
        continue
    labels = []
    for i in range(len(ds)):
        # The window dataset returns (x, label, speed)
        _, label, _ = ds[i]
        labels.append(label.item() if isinstance(label, torch.Tensor) else label)
    labels = torch.tensor(labels)
    print(f"Split: {split_name}")
    print(f" - min: {labels.min().item()}, max: {labels.max().item()}")
    print(f" - unique: {labels.unique(sorted=True).tolist()}")
    # Example: check for out-of-bounds
    num_classes = getattr(data_module, "num_classes", None)
    if num_classes is not None:
        oob = (labels < 0) | (labels >= num_classes)
        if oob.any():
            print(f" - OUT OF BOUNDS: {labels[oob]}")
    print()


Split: train
 - min: 0, max: 2
 - unique: [0, 1, 2]

Split: val
 - min: 0, max: 2
 - unique: [0, 1, 2]

Split: test
 - min: 0, max: 2
 - unique: [0, 1, 2]

